In [ ]:
# Cloning the repository in the notebook

import os

REPO_OWNER = "UtkarshP879"
REPO_NAME = "LSSP_Project_Group_14"
BRANCH_NAME = "main"

REPO_URL = f"https://github.com/{REPO_OWNER}/{REPO_NAME}.git"

if not os.path.exists(REPO_NAME):
  !git clone -b {BRANCH_NAME} {REPO_URL}
else:
  print(f"Directory '{REPO_NAME}' already exists.")

os.chdir(REPO_NAME)
print(f"Current working directory changed to: {os.getcwd()}")

#**Data Loading**

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import soundfile
from IPython.display import Audio, display

guitar_original, sf_guitar = soundfile.read("/content/LSSP_Project_Group_14/Audio/Original_Recordings/guitar_track.wav")
drums_original, sf_drums = soundfile.read("/content/LSSP_Project_Group_14/Audio/Original_Recordings/drums_track.wav")
guitar_and_drums_original, sf_mix = soundfile.read("/content/LSSP_Project_Group_14/Audio/Original_Recordings/guitar_and_drum_track.wav")

print(f"Guitar: Sampling Frequency = {sf_guitar} Hz, Duration = {len(guitar_original) / sf_guitar:.2f} s")
print(f"Drums: Sampling Frequency = {sf_drums} Hz, Duration = {len(drums_original) / sf_drums:.2f} s")
print(f"Guitar and Drums: Sampling Frequency = {sf_mix} Hz, Duration = {len(guitar_and_drums_original) / sf_mix:.2f} s")

if guitar_original.ndim == 2:
  x = guitar_original[:, 0].copy()  # left channel
else:
  x = guitar_original.copy()

print("Using the guitar's left channel for visualization and processing.")

#**Visualizing the Original Signal**
The horizontal axis is time and the vertical axis is sample amplitude.

In [ ]:
sf = sf_guitar
time = np.arange(len(x)) / sf

plt.figure(figsize=(12, 4))
plt.plot(time, x, linewidth=0.6)
plt.title("Original Guitar Audio")
plt.xlabel("Time (s)")
plt.ylabel("Amplitude")
plt.grid(True)
plt.tight_layout()
plt.show()

#**Experiment 1: Room Reverberation Using an LTI System**
#**Constructing the Impulse Response**
We model the room as
$$h[n] = \delta[n] + \sum_{i=1}^{M} a_i \delta[n-d_i]$$

The first term represents the direct sound. Each remaining term represents a delayed reflection. The delays and amplitudes below are chosen values for this simulation.

In [ ]:
# Custom function to make a reverberation impulse response h
# given a list of delays and amplitudes (d(i)s and a(i)s)

def make_impulse_response(delays_ms, amplitudes, sf):
  delay_samples = [round(t * sf / 1000) for t in delays_ms]
  h_case = np.zeros(max(delay_samples) + 1)
  h_case[0] = 1.0
  for d, a in zip(delay_samples, amplitudes):
    h_case[d] += a
  return h_case

In [ ]:
delays_ms = [30, 60, 100]
amps = [0.60, 0.35, 0.20]

delays = [round(t * sf / 1000) for t in delays_ms]
h = make_impulse_response(delays_ms, amps, sf)

print("Sampling frequency:", sf, "Hz")
print("Intended delays (in ms):", delays_ms)
print("Actual delays (in samples):", delays)
print("Impulse response length:", len(h), "samples")

nonzero = np.nonzero(h)[0]
plt.figure(figsize=(10, 4))
plt.stem(nonzero / sf * 1000, h[nonzero])

plt.title("Room Impulse Response (Room Reverberation)")
plt.xlabel("Time (ms)")
plt.ylabel("Amplitude")
plt.grid(True)
plt.tight_layout()
# plt.savefig("experiment1_impulse_response.png", dpi=150)
plt.show()

#**Processing and Visualizing the Signal**
The convolution for this impulse response can be written as a sum of delayed versions of the input signal. Because only non-zero samples of h are used, only these samples will be iterated through in the loop below.

In [ ]:
# Custom function to convolute x with h

def manual_convolution(x, h):
  # Convolve x with h by adding shifted, scaled copies of x
  y = np.zeros(len(x) + len(h) - 1)
  for k in np.nonzero(h)[0]:
      y[k:k + len(x)] += h[k] * x
  return y

In [ ]:
# Convoluting x with h to get the reverberated signal, y

y = manual_convolution(x, h)

print("Input samples:", len(x))
print("Output samples:", len(y))
print(f"Input duration: {len(x) / sf:.3f} s")
print(f"Output duration: {len(y) / sf:.3f} s")

peak = np.max(np.abs(y))
y_audio = 0.9 * y / peak if peak > 0 else y.copy()

print()
print("Original guitar")
display(Audio(x, rate=sf))
print("Reverberated guitar")
display(Audio(y_audio, rate=sf))

In [ ]:
# Comparing the waveforms of the original vs reverberated guitar
# The reverberated signal extends beyond the original because the
# last delayed copy continues after the input ends. Overlapping copies
# can also make some peaks larger.

plt.figure(figsize=(12, 4))
plt.plot(np.arange(len(x)) / sf, x, label="Original", linewidth=0.6, color='blue')
plt.plot(np.arange(len(y_audio)) / sf, y_audio, label="Reverberated", linewidth=0.6, alpha=0.75, color='yellow')

plt.title("Original and Reverberated Guitar")
plt.xlabel("Time (s)")
plt.ylabel("Amplitude")
plt.legend()
plt.grid(True)
plt.tight_layout()
# plt.savefig("experiment1_original_vs_reverberated_waveform.png", dpi=150)
plt.show()

#**Experimenting with Impulse Response Parameters**


In [ ]:
# Experimenting with different cases

cases = {
    "Baseline": ([30, 60, 100], [0.60, 0.35, 0.20]),
    "Fewer_reflections": ([30, 60], [0.60, 0.35]),
    "Longer_delays": ([60, 120, 200], [0.60, 0.35, 0.20]),
    "Weaker_reflections": ([30, 60, 100], [0.20, 0.10, 0.05])
}

reverb_results = {}
for name, (case_delays, case_amps) in cases.items():
  h_case = make_impulse_response(case_delays, case_amps, sf)
  y_case = manual_convolution(x, h_case)
  y_case_audio = 0.9 * y_case / np.max(np.abs(y_case)) if np.max(np.abs(y_case)) > 0 else y_case.copy()
  # soundfile.write(f"exp1_reverberated_guitar_{name}.wav", y_case_audio, sf)

  reverb_results[name] = y_case_audio

  print(f"{name}: {len(case_delays)} reflections")
  print(f"Output duration: {len(y_case_audio) / sf:.3f} s")
  display(Audio(y_case_audio, rate=sf))

## Observations

- **Baseline:** The original signal is combined with three delayed reflections of decreasing amplitudes, producing a reverberation effect.
- **Fewer reflections:** Reducing the number of reflections reduces the number of delayed copies added to the original signal.
- **Longer delays:** Increasing the reflection delays makes the reflected sounds arrive later, making them more separated from the direct sound.
- **Weaker reflections:** Reducing the reflection amplitudes makes the delayed copies quieter, resulting in a less pronounced reverberation effect.

#**Calculating and Comparing the Magnitude Spectra**


In [ ]:
# Custom recursive function to calculate the Fast-Fourier Transform (FFT)

def manual_fft(signal):
  signal = np.asarray(signal, dtype=complex)
  n = len(signal)

  if n == 1:
    return signal

  if n & (n - 1) != 0:
    raise ValueError("Input length must be a power of two.")

  even = manual_fft(signal[::2])
  odd = manual_fft(signal[1::2])

  factor = np.exp(-2j * np.pi * np.arange(n // 2) / n) * odd

  return np.concatenate((even + factor, even - factor))

In [ ]:
# Custom function to calculate the magnitude spectrum of a signal using the FFT

def spectrum_db(signal, nfft):
  padded = np.zeros(nfft)
  padded[:len(signal)] = signal

  S = np.abs(manual_fft(padded)[:nfft // 2])
  return 20 * np.log10(S / (S.max() + 1e-12) + 1e-12)

In [ ]:
# Calculting the spectra for the original guitar signal

nfft = 2 ** int(np.ceil(np.log2(max(len(x), len(y_audio)))))
freq = np.arange(nfft // 2) * sf / nfft

X_db = spectrum_db(x, nfft)

print("FFT length:", nfft)

In [ ]:
# Comparing the spectra of the original signal against
# the baseline case

plt.figure(figsize=(11, 4.5))
Y_baseline_db = spectrum_db(reverb_results["Baseline"], nfft)
plt.plot(freq, X_db, label="Original", linewidth=0.7, color='blue')
plt.plot(freq, Y_baseline_db, label="Reverberated (Baseline)", linewidth=0.7, alpha=0.8, color='yellow')
plt.xlim(0, 10000)
plt.ylim(-130, 5)
plt.xlabel("Frequency (Hz)")
plt.ylabel("Magnitude (dB, relative to peak)")
plt.title(f"Magnitude Spectra: Original vs Reverberated (Baseline)")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
"""
# Comparing the spectra of the original signal against
# all the modified signals

plt.figure(figsize=(11, 18))

for i, (name, y_case_audio) in enumerate(reverb_results.items(), start=1):
  Y_case_db = spectrum_db(y_case_audio, nfft)
  plt.subplot(4, 1, i)
  plt.plot(freq, X_db, label="Original", linewidth=0.7, color='blue')
  plt.plot(freq, Y_case_db, label=name, linewidth=0.7, alpha=0.8, color='yellow')
  plt.xlim(0, 10000)
  plt.ylim(-130, 5)
  plt.xlabel("Frequency (Hz)")
  plt.ylabel("Magnitude (dB, relative to peak)")
  plt.title(f"Magnitude Spectra: Original vs Reverberated ({name})")
  plt.legend()
  plt.grid(True)

plt.tight_layout()
plt.savefig("experiment1_magnitude_spectra.png", dpi=150)
plt.show()
"""

#**Experiment 2: Musical Effects Using LTV Systems**
#**Formulating the Periodic Delay**
We model the periodic delay effect as
$$y[n] = x[n] + \alpha x[n - D[n]]$$

The first term represents the direct sound. The second term represents the delay. The delay D[n] is modeled as:
$$D[n] = D_0 + \Delta Dsin(\frac{2\pi f_mn}{f_s})$$

This delay is periodic when $$\frac{f_m}{f_s}$$ is a rational number in the least form $$\frac{p}{q}$$ where its period will then be q.

#**Processing and Visualizing the Signal**
The output signal must be calculated sample by sample, with the delay term $$\alpha x[n - D[n]]$$ only coming into play when n - D[n] is non-negative.

In [ ]:
# Custom function to calculate y from x

def ltv_system_output(x, alpha, D_0, delta_D, mf, sf):
  # Calculate the delay term sample-by-sample:
  delay_term = np.zeros(len(x))
  for n in range(len(x)):
    D = D_0 + delta_D * np.sin(2 * np.pi * mf * n / sf)
    if (n - int(D)) >= 0 and ((n - int(D)) <= n):
      delay_term[n] = alpha * x[n - int(D)]
  return x + delay_term

In [ ]:
# Calculating the output y and comparing it to the
# fixed-delay echo from Experiment 1

ALPHA = 0.8
D_0 = 200 # in ms
D_0 = D_0 * sf / 1000 # in samples
DELTA_D = 10 # in ms
DELTA_D = DELTA_D * sf / 1000 # in samples
MF = 0.2

y = ltv_system_output(x, ALPHA, D_0, DELTA_D, MF, sf)

print("Input samples:", len(x))
print("Output samples:", len(y))
print(f"Input duration: {len(x) / sf:.3f} s")
print(f"Output duration: {len(y) / sf:.3f} s")

peak = np.max(np.abs(y))
y_audio = 0.9 * y / peak if peak > 0 else y.copy()

fixed_delay_guitar, sf_fixed_delay = soundfile.read("/content/LSSP_Project_Group_14/Audio/Experiment_Outputs/exp1_reverberated_guitar_Baseline.wav")

print()
print("Original guitar")
display(Audio(x, rate=sf))
print("Delay Effect guitar")
display(Audio(y_audio, rate=sf))
print("Fixed-delay echo guitar (from Experiment 1)")
display(Audio(fixed_delay_guitar, rate=sf))

In [ ]:
# Comparing the waveforms of the original vs delay effect guitar
# Overlapping peaks can make some peaks larger

plt.figure(figsize=(12, 4))
plt.plot(np.arange(len(x)) / sf, x, label="Original", linewidth=0.6, color='blue')
plt.plot(np.arange(len(y_audio)) / sf, y_audio, label="Delay Effect", linewidth=0.6, alpha=0.75, color='yellow')

plt.title("Original and Delay Effect Guitar")
plt.xlabel("Time (s)")
plt.ylabel("Amplitude")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

#**Experimenting with Impulse Response Parameters**


In [ ]:
# Experimenting with different cases

cases = {
    "Baseline": (0.8, 200, 10, 0.2),
    "No_delay_variation": (0.8, 200, 0, 0.2),
    "Higher_variation": (0.8, 200, 20, 0.2),
    "Higher_modulation": (0.8, 200, 10, 1)
}

delay_results = {}
for name, (alpha, D_0, delta_D, mf) in cases.items():
  y_case = ltv_system_output(x, alpha, ((D_0*sf)/1000), ((delta_D*sf)/1000), mf, sf)
  y_case_audio = 0.9 * y_case / np.max(np.abs(y_case)) if np.max(np.abs(y_case)) > 0 else y_case.copy()
  # soundfile.write(f"exp2_delay_effect_guitar_{name}.wav", y_case_audio, sf)

  delay_results[name] = y_case_audio

  print(f"{name} case:")
  print(f"Output duration: {len(y_case_audio) / sf:.3f} s")
  display(Audio(y_case_audio, rate=sf))

In [ ]:
# Comparing the waveforms of the various outputs with
# the original signal

plt.figure(figsize=(11, 18))
for i, (name, y_case_audio) in enumerate(delay_results.items(), start=1):
  plt.subplot(4, 1, i)
  plt.plot(np.arange(len(x)) / sf, x, label="Original", linewidth=0.6, color='blue')
  plt.plot(np.arange(len(y_case_audio)) / sf, y_case_audio, label=name, linewidth=0.6, alpha=0.75, color='yellow')

  plt.title(f"Waveform: Original vs Delay Effect ({name})")
  plt.xlabel("Time (s)")
  plt.ylabel("Amplitude")
  plt.legend()
  plt.grid(True)

plt.tight_layout()
# plt.savefig("experiment2_waveforms.png", dpi=150)
plt.show()

## Observations

- **Baseline:** The output has a distinct delay effect, along with a chorus effect of multiple copies of the same instrument playing close to each other.
- **No Delay Variation:** A clear distinctive constant delay can be noticed in the output, without the chorus effect seen in the baseline.
- **Higher Variation:** The chorus effect of multiple simultaneous instruments is further exxagerated. Slight pitch distortion is also apparent due to non-linear phase interactions between delayed copies.
- **Higher Modulation:** Severe pitch distortion can be seen along with a distinct "sweeping" effect produced by the faster oscillations in the delay.

#**Calculating and Comparing the Magnitude Spectra**


In [ ]:
# Comparing the spectra of the original signal against
# the baseline case

plt.figure(figsize=(11, 4.5))
Y_baseline_db = spectrum_db(delay_results["Baseline"], nfft)
plt.plot(freq, X_db, label="Original", linewidth=0.7, color='blue')
plt.plot(freq, Y_baseline_db, label="Reverberated (Baseline)", linewidth=0.7, alpha=0.8, color='yellow')
plt.xlim(0, 20000)
plt.ylim(-130, 5)
plt.xlabel("Frequency (Hz)")
plt.ylabel("Magnitude (dB, relative to peak)")
plt.title(f"Magnitude Spectra: Original vs Delay Effect (Baseline)")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
"""
# Comparing the spectra of the original signal against
# all the modified signals

plt.figure(figsize=(11, 18))

for i, (name, y_case_audio) in enumerate(delay_results.items(), start=1):
  Y_case_db = spectrum_db(y_case_audio, nfft)
  plt.subplot(4, 1, i)
  plt.plot(freq, X_db, label="Original", linewidth=0.7, color='blue')
  plt.plot(freq, Y_case_db, label=name, linewidth=0.7, alpha=0.8, color='yellow')
  plt.xlim(0, 20000)
  plt.ylim(-130, 5)
  plt.xlabel("Frequency (Hz)")
  plt.ylabel("Magnitude (dB, relative to peak)")
  plt.title(f"Magnitude Spectra: Original vs Delay Effect ({name})")
  plt.legend()
  plt.grid(True)

plt.tight_layout()
# plt.savefig("experiment2_magnitude_spectra.png", dpi=150)
plt.show()
"""